# Data Discovery

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from glob import glob
from PIL import Image
import rasterio
import torch
from torch.utils.data import Dataset, DataLoader

In [ ]:
class WaterDataset(Dataset):
    def __init__(self, image_dir, label_dir, transform=None):
        # Only take images that have a corresponding label with the same name (e.g., 0.tif -> 0.png)
        self.image_paths = sorted(glob(os.path.join(image_dir, "*.tif")))
        self.label_paths = []
        
        valid_image_paths = []
        for img_path in self.image_paths:
            base_name = os.path.basename(img_path).split('.')[0]
            label_path = os.path.join(label_dir, f"{base_name}.png")
            if os.path.exists(label_path):
                valid_image_paths.append(img_path)
                self.label_paths.append(label_path)
        
        self.image_paths = valid_image_paths
        self.transform = transform
        print(f"Found {len(self.image_paths)} valid image-label pairs.")
        if len(self.image_paths) > 0:
            print("Sample matches:")
            for i in range(min(3, len(self.image_paths))):
                print(f"  Image: {os.path.basename(self.image_paths[i])} <-> Label: {os.path.basename(self.label_paths[i])}")


    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        # Load multispectral image
        with rasterio.open(self.image_paths[idx]) as src:
            img = src.read().astype(np.float32)  # shape: (bands, H, W)

        # Load label
        label = np.array(Image.open(self.label_paths[idx]))  # shape: (H, W)
        label = (label > 0).astype(np.float32) # Ensure binary

        if self.transform:
            img, label = self.transform(img, label)

        return torch.tensor(img, dtype=torch.float32), torch.tensor(label, dtype=torch.float32)

In [ ]:
def compute_dataset_stats(loader):
    """Compute global mean and std for each band."""
    sum_ = torch.zeros(12)
    sum_sq = torch.zeros(12)
    total_pixels = 0

    for imgs, _ in loader:
        # imgs shape: [B, 12, H, W]
        B, C, H, W = imgs.shape
        imgs = imgs.view(B, C, -1)
        sum_ += imgs.sum(dim=[0, 2])
        sum_sq += (imgs**2).sum(dim=[0, 2])
        total_pixels += B * H * W
    
    mean = sum_ / total_pixels
    std = torch.sqrt((sum_sq / total_pixels) - (mean**2) + 1e-6)
    return mean, std

In [ ]:
data_dir = "/kaggle/input/datasets/mennaezzelarab/water-segmentation-dataset/data"
image_dir = os.path.join(data_dir, "images")
label_dir = os.path.join(data_dir, "labels")

dataset = WaterDataset(image_dir, label_dir)

In [ ]:
def plot_multispectral_bands(img, title=""):
    bands_names = [
        "Coastal Aerosol", "Blue", "Green", "Red", "NIR",
        "SWIR1", "SWIR2", "QA Band", "Merit DEM",
        "Copernicus DEM", "ESA World Cover", "Water Occurrence Probability"
    ]

    n_bands = img.shape[0]
    plt.figure(figsize=(20, 10))
    for i in range(n_bands):
        plt.subplot(3, 4, i+1)
        plt.imshow(img[i], cmap='gray')
        plt.title(bands_names[i])
        plt.axis('off')
    plt.suptitle(title)
    plt.show()

# Example visualization for the first sample
img, label = dataset[0]
plot_multispectral_bands(img.numpy(), title="Image 0 Multispectral Bands")

plt.figure(figsize=(6,6))
plt.imshow(label.numpy(), cmap='Blues')
plt.title("Ground Truth Mask")
plt.axis('off')
plt.show()

In [ ]:
def plot_rgb_only(img):

    rgb = np.stack([img[3], img[2], img[1]], axis=-1)

    # Normalize for visualization
    rgb = (rgb - rgb.min()) / (rgb.max() - rgb.min())

    plt.figure(figsize=(6,6))
    plt.imshow(rgb)
    plt.title("RGB Composite")
    plt.axis('off')
    plt.show()

# Example for first image
img, _ = dataset[0]
plot_rgb_only(img.numpy())

In [ ]:
def plot_rgb(img, label=None):
    rgb = np.stack([img[3], img[2], img[1]], axis=-1)

    # Normalize for visualization
    rgb = (rgb - rgb.min()) / (rgb.max() - rgb.min())

    plt.figure(figsize=(6,6))
    plt.imshow(rgb)
    if label is not None:
        plt.imshow(label, cmap='Blues', alpha=0.3)  # overlay mask
    plt.axis('off')
    plt.show()

plot_rgb(img.numpy(), label.numpy())

In [ ]:
def plot_rgb_grid(dataset, indices=None, n_cols=4):
    """
    Display multiple RGB composites in a grid.

    Args:
        dataset: PyTorch dataset
        indices: list of indices to display (default: first n images)
        n_cols: number of columns in the grid
    """
    if indices is None:
        indices = list(range(min(8, len(dataset))))  # default: first 8 images

    n_rows = (len(indices) + n_cols - 1) // n_cols
    plt.figure(figsize=(4*n_cols, 4*n_rows))

    for i, idx in enumerate(indices):
        img, _ = dataset[idx]
        rgb = np.stack([img[3], img[2], img[1]], axis=-1)
        rgb = (rgb - rgb.min()) / (rgb.max() - rgb.min())

        plt.subplot(n_rows, n_cols, i+1)
        plt.imshow(rgb)
        plt.title(f"Image {idx}")
        plt.axis('off')

    plt.tight_layout()
    plt.show()

# Example: visualize first 12 RGB images in a grid
plot_rgb_grid(dataset, indices=list(range(12)), n_cols=4)

# Preprocessing

In [ ]:
from torch.utils.data import Subset, random_split

# Total number of samples
num_samples = len(dataset)

# Compute split sizes
train_size = int(0.7 * num_samples)
val_size = test_size = int(0.15 * num_samples)  # 45 each
train_size = num_samples - val_size - test_size  # 216

# Perform split
train_dataset, val_dataset, test_dataset = random_split(
    dataset, [train_size, val_size, test_size],
    generator=torch.Generator().manual_seed(42)  # reproducible
)

print(f"Train: {len(train_dataset)}, Val: {len(val_dataset)}, Test: {len(test_dataset)}")

In [ ]:
batch_size = 8

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

In [ ]:
# Backup original transform
original_transform = train_dataset.dataset.transform  # if you wrapped Subset around base dataset

# Remove transform
train_dataset.dataset.transform = None

class NormalizedWaterDataset(Dataset):
    def __init__(self, base_dataset, mean, std):
        self.base_dataset = base_dataset
        self.mean = mean.view(12, 1, 1)
        self.std = std.view(12, 1, 1)

    def __len__(self):
        return len(self.base_dataset)

    def __getitem__(self, idx):
        img, label = self.base_dataset[idx]
        # Normalize: (x - mean) / std
        img = (img - self.mean) / (self.std + 1e-8)
        return img, label

# Remove existing split logic to re-calculate correctly
train_dataset, val_dataset, test_dataset = random_split(
    dataset, [train_size, val_size, test_size],
    generator=torch.Generator().manual_seed(42)
)

In [ ]:
# Temporarily use base train_dataset to compute stats
temp_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=False)
train_mean, train_std = compute_dataset_stats(temp_loader)
print("Global Mean:", train_mean)
print("Global Std:", train_std)

# Apply normalization
train_dataset_norm = NormalizedWaterDataset(train_dataset, train_mean, train_std)
val_dataset_norm = NormalizedWaterDataset(val_dataset, train_mean, train_std)
test_dataset_norm = NormalizedWaterDataset(test_dataset, train_mean, train_std)

train_loader = DataLoader(train_dataset_norm, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset_norm, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset_norm, batch_size=batch_size, shuffle=False)

In [ ]:
for imgs, labels in train_loader:
    print(imgs.shape, labels.shape)  # should be (batch_size, 12, 128, 128), (batch_size, 128, 128)
    # Move to CPU for visualization
    plot_rgb_only(imgs[0].cpu().numpy())
    break

# Model Architecture

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class UNet(nn.Module):
    def __init__(self, in_channels=12, out_channels=1, features=[64, 128, 256, 512]):
        super(UNet, self).__init__()
        self.downs = nn.ModuleList()
        self.ups = nn.ModuleList()
        self.pool = nn.MaxPool2d(2)

        # Downsampling
        for feature in features:
            self.downs.append(self.double_conv(in_channels, feature))
            in_channels = feature

        # Bottleneck
        self.bottleneck = self.double_conv(features[-1], features[-1]*2)

        # Upsampling
        for feature in reversed(features):
            self.ups.append(nn.ConvTranspose2d(feature*2, feature, kernel_size=2, stride=2))
            self.ups.append(self.double_conv(feature*2, feature))

        # Final conv
        self.final_conv = nn.Conv2d(features[0], out_channels, kernel_size=1)

    def forward(self, x):
        skip_connections = []

        for down in self.downs:
            x = down(x)
            skip_connections.append(x)
            x = self.pool(x)

        x = self.bottleneck(x)
        skip_connections = skip_connections[::-1]

        for idx in range(0, len(self.ups), 2):
            x = self.ups[idx](x)  # upsample
            skip_connection = skip_connections[idx//2]
            if x.shape != skip_connection.shape:
                x = F.interpolate(x, size=skip_connection.shape[2:])
            x = torch.cat((skip_connection, x), dim=1)
            x = self.ups[idx+1](x)  # conv

        return self.final_conv(x)

    @staticmethod
    def double_conv(in_channels, out_channels):
        return nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
        )

# Instantiate model
model = UNet(in_channels=12, out_channels=1)
print(model)

In [ ]:
from torchsummary import summary

# Move model to device
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

# Print summary for input size [12, 128, 128]
summary(model, input_size=(12, 128, 128))

In [ ]:
import torch
import torch.nn as nn

# Loss function
criterion = nn.BCEWithLogitsLoss()

# Threshold for converting logits to binary masks
threshold = 0.5

# Metric functions
def compute_iou(preds, labels, threshold=0.5):
    preds = (torch.sigmoid(preds) > threshold).float()
    intersection = (preds * labels).sum(dim=(1,2))
    union = (preds + labels - preds*labels).sum(dim=(1,2))
    iou = (intersection + 1e-6) / (union + 1e-6)
    return iou.mean().item()

def compute_f1(preds, labels, threshold=0.5):
    preds = (torch.sigmoid(preds) > threshold).float()
    tp = (preds * labels).sum(dim=(1,2))
    fp = (preds * (1 - labels)).sum(dim=(1,2))
    fn = ((1 - preds) * labels).sum(dim=(1,2))
    f1 = 2 * tp / (2*tp + fp + fn + 1e-6)
    return f1.mean().item()

def compute_precision_recall(preds, labels, threshold=0.5):
    preds = (torch.sigmoid(preds) > threshold).float()
    tp = (preds * labels).sum(dim=(1,2))
    fp = (preds * (1 - labels)).sum(dim=(1,2))
    fn = ((1 - preds) * labels).sum(dim=(1,2))
    precision = (tp + 1e-6) / (tp + fp + 1e-6)
    recall = (tp + 1e-6) / (tp + fn + 1e-6)
    return precision.mean().item(), recall.mean().item()

In [ ]:
class DiceLoss(nn.Module):
    def __init__(self, smooth=1e-6):
        super(DiceLoss, self).__init__()
        self.smooth = smooth

    def forward(self, preds, targets):
        preds = torch.sigmoid(preds)
        
        # Flatten label and prediction tensors
        preds = preds.view(-1)
        targets = targets.view(-1)
        
        intersection = (preds * targets).sum()
        dice = (2. * intersection + self.smooth) / (preds.sum() + targets.sum() + self.smooth)
        
        return 1 - dice

In [ ]:
import torch.optim as optim

# Optimizer
optimizer = optim.Adam(model.parameters(), lr=1e-3)

# Device
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    running_loss = 0.0
    running_iou = 0.0
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device).unsqueeze(1)
        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item() * imgs.size(0)
        running_iou += compute_iou(outputs.squeeze(1), labels.squeeze(1)) * imgs.size(0)
    
    return running_loss / len(loader.dataset), running_iou / len(loader.dataset)

def validate(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    running_iou = 0.0
    with torch.no_grad():
        for imgs, labels in loader:
            imgs, labels = imgs.to(device), labels.to(device).unsqueeze(1)
            outputs = model(imgs)
            loss = criterion(outputs, labels)
            running_loss += loss.item() * imgs.size(0)
            running_iou += compute_iou(outputs.squeeze(1), labels.squeeze(1)) * imgs.size(0)
            
    return running_loss / len(loader.dataset), running_iou / len(loader.dataset)

# Stage 1: BCE Training (100 Epochs)

In [ ]:
print("Starting Stage 1: BCE Loss Training...")
pos_weight = torch.tensor([5.0]).to(device) 
criterion_bce = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=10)

num_epochs_stage1 = 100
best_val_iou = 0.0
history = {"train_loss": [], "train_iou": [], "val_loss": [], "val_iou": []}

for epoch in range(num_epochs_stage1):
    train_loss, train_iou = train_one_epoch(model, train_loader, optimizer, criterion_bce, device)
    val_loss, val_iou = validate(model, val_loader, criterion_bce, device)
    scheduler.step(val_iou)
    
    history["train_loss"].append(train_loss)
    history["train_iou"].append(train_iou)
    history["val_loss"].append(val_loss)
    history["val_iou"].append(val_iou)
    
    if val_iou > best_val_iou:
        best_val_iou = val_iou
        torch.save(model.state_dict(), "best_unet_bce.pth")
    
    if (epoch + 1) % 10 == 0:
        print(f"Epoch [{epoch+1}/{num_epochs_stage1}] Train Loss: {train_loss:.4f}, IoU: {train_iou:.4f} | Val Loss: {val_loss:.4f}, IoU: {val_iou:.4f}")

# Stage 2: Dice Loss Fine-tuning

In [ ]:
print("Starting Stage 2: Dice Loss Fine-tuning...")
model.load_state_dict(torch.load("best_unet_bce.pth"))
criterion_dice = DiceLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)
num_epochs_stage2 = 50

for epoch in range(num_epochs_stage2):
    train_loss, train_iou = train_one_epoch(model, train_loader, optimizer, criterion_dice, device)
    val_loss, val_iou = validate(model, val_loader, criterion_dice, device)
    
    history["train_loss"].append(train_loss)
    history["train_iou"].append(train_iou)
    history["val_loss"].append(val_loss)
    history["val_iou"].append(val_iou)
    
    if val_iou > best_val_iou:
        best_val_iou = val_iou
        torch.save(model.state_dict(), "best_unet_final.pth")
    
    if (epoch + 1) % 10 == 0:
        print(f"Epoch [{epoch+1}/{num_epochs_stage2}] Train Loss: {train_loss:.4f}, IoU: {train_iou:.4f} | Val Loss: {val_loss:.4f}, IoU: {val_iou:.4f}")

In [ ]:
def plot_metrics(history):
    epochs = range(1, len(history["train_loss"]) + 1)
    plt.figure(figsize=(12, 5))
    
    plt.subplot(1, 2, 1)
    plt.plot(epochs, history["train_loss"], label='Train Loss')
    plt.plot(epochs, history["val_loss"], label='Val Loss')
    plt.axvline(x=100, color='r', linestyle='--', label='Stage 2 Start')
    plt.title('Loss History')
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.legend()
    
    plt.subplot(1, 2, 2)
    plt.plot(epochs, history["train_iou"], label='Train IoU')
    plt.plot(epochs, history["val_iou"], label='Val IoU')
    plt.axvline(x=100, color='r', linestyle='--', label='Stage 2 Start')
    plt.title('IoU History')
    plt.xlabel('Epochs')
    plt.ylabel('IoU')
    plt.legend()
    
    plt.tight_layout()
    plt.show()

plot_metrics(history)

In [ ]:
def final_evaluation(model, loader, device):
    model.eval()
    all_precision = []
    all_recall = []
    all_f1 = []
    all_iou = []
    
    with torch.no_grad():
        for imgs, labels in loader:
            imgs, labels = imgs.to(device), labels.to(device)
            outputs = model(imgs).squeeze(1)
            
            precision, recall = compute_precision_recall(outputs, labels)
            f1 = compute_f1(outputs, labels)
            iou = compute_iou(outputs, labels)
            
            all_precision.append(precision)
            all_recall.append(recall)
            all_f1.append(f1)
            all_iou.append(iou)
            
    print("\n" + "="*30)
    print("FINAL TEST SET EVALUATION")
    print("="*30)
    print(f"Precision: {np.mean(all_precision):.4f}")
    print(f"Recall:    {np.mean(all_recall):.4f}")
    print(f"F1-Score:  {np.mean(all_f1):.4f}")
    print(f"IoU:       {np.mean(all_iou):.4f}")
    print("="*30)

In [ ]:
model.load_state_dict(torch.load("best_unet_final.pth"))
final_evaluation(model, test_loader, device)
model.eval()

In [ ]:
def visualize_prediction(img_tensor, label_tensor, model, threshold=0.5):
    model.eval()
    with torch.no_grad():
        img = img_tensor.unsqueeze(0).to(device)  # Add batch dim
        output = model(img)
        pred_mask = torch.sigmoid(output.squeeze(0))  # [1,H,W]
        pred_mask = (pred_mask > threshold).cpu().numpy().squeeze()  # now [H,W]

    # RGB composite from Red (3), Green (2), Blue (1) bands
    img_rgb = img_tensor[[3,2,1],:,:].cpu().numpy()  # R,G,B
    # DEBUG: Print range to verify normalization
    print(f"Visualization Input Range - Min: {img_tensor.min():.4f}, Max: {img_tensor.max():.4f}")
    
    img_rgb = (img_rgb - img_rgb.min()) / (img_rgb.max() - img_rgb.min() + 1e-8)  # normalize for display
    img_rgb = np.transpose(img_rgb, (1,2,0))  # H,W,C for imshow

    fig, axs = plt.subplots(1,3, figsize=(12,4))
    axs[0].imshow(img_rgb)
    axs[0].set_title("RGB Composite")
    axs[0].axis('off')

    axs[1].imshow(label_tensor.cpu().numpy(), cmap='Blues')
    axs[1].set_title("Ground Truth")
    axs[1].axis('off')

    axs[2].imshow(pred_mask, cmap='Blues')
    axs[2].set_title("Predicted Mask")
    axs[2].axis('off')

    plt.show()

# Example: visualize first image in test dataset
# CRITICAL: Use normalized dataset for visualization!
test_img, test_label = test_dataset_norm[5] 
visualize_prediction(test_img, test_label, model)

In [ ]:
def visualize_multiple_predictions(dataset, model, indices, threshold=0.5, ncols=3):
    model.eval()
    n = len(indices)
    nrows = int(np.ceil(n / ncols))
    fig, axs = plt.subplots(nrows, ncols*3, figsize=(ncols*4, nrows*4))
    axs = axs.flatten()

    for i, idx in enumerate(indices):
        img_tensor, label_tensor = dataset[idx]
        with torch.no_grad():
            img = img_tensor.unsqueeze(0).to(device)
            output = model(img)
            pred_mask = torch.sigmoid(output.squeeze(0))
            pred_mask = (pred_mask > threshold).cpu().numpy().squeeze()

        # RGB composite Red (3), Green (2), Blue (1)
        img_rgb = img_tensor[[3,2,1],:,:].cpu().numpy()
        img_rgb = (img_rgb - img_rgb.min()) / (img_rgb.max() - img_rgb.min() + 1e-8)
        img_rgb = np.transpose(img_rgb, (1,2,0))

        axs[i*3].imshow(img_rgb)
        axs[i*3].set_title(f"RGB {idx}")
        axs[i*3].axis('off')

        axs[i*3+1].imshow(label_tensor.cpu().numpy(), cmap='Blues')
        axs[i*3+1].set_title(f"GT {idx}")
        axs[i*3+1].axis('off')

        axs[i*3+2].imshow(pred_mask, cmap='Blues')
        axs[i*3+2].set_title(f"Pred {idx}")
        axs[i*3+2].axis('off')

    for j in range(n*3, len(axs)):
        axs[j].axis('off')

    plt.tight_layout()
    plt.show()

# Example: visualize multiple test samples from normalized set
visualize_multiple_predictions(test_dataset_norm, model, indices=list(range(15)), ncols=3)